# House Price Predictor — Google Colab Walkthrough

**Lumexa Data Scientist course**

A complete, real machine-learning regression project: predict the median house value
in a California district from real 1990 U.S. census data. This notebook walks through
the full workflow — clean data, engineer features, train multiple models, evaluate
honestly, tune hyperparameters, and use the trained model to make predictions (in place
of running a separate API server).

This notebook is fully self-contained and works with **Runtime → Run all** — no setup,
no API keys, no accounts, and no files to upload. The dataset is downloaded directly from
a public GitHub URL at runtime.

**What you'll do:**
1. Download and inspect the real California Housing dataset
2. Clean missing values and engineer new features
3. Train a Linear Regression baseline and a Random Forest model
4. Evaluate both with MAE / MSE / RMSE / R²
5. Tune the Random Forest with `RandomizedSearchCV`
6. Pick the best model and use it to predict prices for new houses (no Flask server needed)


In [1]:
# scikit-learn, pandas, numpy are preinstalled in Google Colab.
# joblib ships with scikit-learn, so nothing extra needs installing here.
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Libraries loaded.")


Libraries loaded.


## Step 1: Download and load the dataset

**California Housing** (1990 U.S. census, originally distributed via StatLib, popularized
by Aurélien Géron's *Hands-On Machine Learning*).

- Source URL: `https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv`
- License: public domain / freely redistributable census-derived data.
- Rows: 20,640. Columns: 10 — `longitude, latitude, housing_median_age, total_rooms,
  total_bedrooms, population, households, median_income, median_house_value,
  ocean_proximity`.

We download it directly from GitHub, so there's nothing to upload manually.


In [2]:
DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"

df = pd.read_csv(DATA_URL)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns.")
df.head()


Loaded 20640 rows, 10 columns.


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Step 2: Inspect data quality

Real data is never perfectly clean. Let's measure the actual quality issues in this
dataset directly, rather than assuming they exist.


In [3]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

print(f"\nDuplicate rows: {df.duplicated().sum()}")

print("\nocean_proximity value counts:")
print(df["ocean_proximity"].value_counts())

print(f"\nmedian_house_value range: ${df['median_house_value'].min():,.0f} - ${df['median_house_value'].max():,.0f}")
print("Note the capped top value ($500,001) — a well-known quirk of this dataset.")


Missing values per column:
total_bedrooms    207
dtype: int64

Duplicate rows: 0

ocean_proximity value counts:
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

median_house_value range: $14,999 - $500,001
Note the capped top value ($500,001) — a well-known quirk of this dataset.


We should see **207 missing values**, all in `total_bedrooms`, and **0 duplicate rows**.
We handle the missing values with a median `SimpleImputer` inside the model pipeline below
(never dropping rows), so no real data is discarded.


## Step 3: Feature engineering

Raw counts like `total_rooms` and `total_bedrooms` are less informative per-household than
ratios. We engineer three new features that a raw model can't infer on its own:

- `rooms_per_household`
- `bedrooms_per_room`
- `population_per_household`


In [4]:
def engineer_features(data):
    data = data.copy()
    data["rooms_per_household"] = data["total_rooms"] / data["households"]
    data["bedrooms_per_room"] = data["total_bedrooms"] / data["total_rooms"]
    data["population_per_household"] = data["population"] / data["households"]
    return data

df = engineer_features(df)
df[["rooms_per_household", "bedrooms_per_room", "population_per_household"]].describe()


,rooms_per_household,bedrooms_per_room,population_per_household
count,20640.000000,20433.000000,20640.000000
mean,5.429000,0.213039,3.070655
std,2.474173,0.057983,10.386050
min,0.846154,0.100000,0.692308
25%,4.440716,0.175427,2.429741
50%,5.229129,0.203162,2.818116
75%,6.052381,0.239821,3.282261
max,141.909091,1.000000,1243.333333


## Step 4: Split into train/test sets

We hold out 20% of the data as a test set the model never sees during training, so our
evaluation metrics reflect real generalization performance.


In [5]:
target = "median_house_value"
y = df[target]
X = df.drop(columns=[target])

categorical_features = ["ocean_proximity"]
numeric_features = [c for c in X.columns if c not in categorical_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")


Train rows: 16512, Test rows: 4128


## Step 5: Build a reusable preprocessing + model pipeline

Numeric columns are median-imputed then standardized; the categorical `ocean_proximity`
column is imputed with the most frequent value then one-hot encoded. Wrapping everything
in a scikit-learn `Pipeline` means the exact same preprocessing is applied at prediction
time later, with no risk of train/test skew.


In [6]:
def build_pipeline(numeric_feats, categorical_feats, model):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_feats),
        ("cat", categorical_pipeline, categorical_feats),
    ])
    return Pipeline([("preprocess", preprocessor), ("model", model)])


def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] MAE={mae:,.2f}  MSE={mse:,.2f}  RMSE={rmse:,.2f}  R2={r2:.4f}")
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

print("Helper functions ready.")


Helper functions ready.


## Step 6: Train a Linear Regression baseline

A simple, fast, interpretable model to compare everything else against.


In [7]:
results = {}

lin_pipeline = build_pipeline(numeric_features, categorical_features, LinearRegression())
lin_pipeline.fit(X_train, y_train)
lin_pred = lin_pipeline.predict(X_test)
results["linear_regression"] = evaluate("LinearRegression", y_test, lin_pred)


[LinearRegression] MAE=49,645.49  MSE=4,778,547,424.03  RMSE=69,127.04  R2=0.6353


## Step 7: Train a Random Forest

Random Forests capture non-linear relationships and feature interactions that a linear
model cannot.


In [8]:
rf_pipeline = build_pipeline(
    numeric_features, categorical_features,
    RandomForestRegressor(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1),
)
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
results["random_forest"] = evaluate("RandomForest (untuned)", y_test, rf_pred)


[RandomForest (untuned)] MAE=31,889.92  MSE=2,461,153,552.23  RMSE=49,610.01  R2=0.8122


## Step 8: Tune the Random Forest with RandomizedSearchCV

We search over a small grid of hyperparameters (10 random combinations, 3-fold
cross-validation) to see if we can beat the untuned forest. This cell does real
cross-validated model selection, so it takes a little while to run — that's expected.


In [9]:
param_dist = {
    "model__n_estimators": [100, 150, 200],
    "model__max_depth": [10, 15, 20, 25],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
}
search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="neg_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)
search.fit(X_train, y_train)
best_rf = search.best_estimator_
best_pred = best_rf.predict(X_test)
results["random_forest_tuned"] = evaluate("RandomForest (tuned)", y_test, best_pred)
results["random_forest_tuned"]["best_params"] = search.best_params_
print("Best params:", search.best_params_)


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[RandomForest (tuned)] MAE=32,968.48  MSE=2,453,324,526.26  RMSE=49,531.05  R2=0.8128
Best params: {'model__n_estimators': 150, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': 20}


## Step 9: Pick the best model

We select the model with the lowest RMSE on the real held-out test set — not the
training set, which would give an overly optimistic picture.


In [10]:
candidates = {
    "linear_regression": lin_pipeline,
    "random_forest": rf_pipeline,
    "random_forest_tuned": best_rf,
}
best_name = min(results, key=lambda k: results[k]["RMSE"])
best_model = candidates[best_name]

print(f"Best model: {best_name} (RMSE={results[best_name]['RMSE']:,.2f}, R2={results[best_name]['R2']:.4f})")
print()
print(f"{'Model':<22}{'MAE':>14}{'MSE':>18}{'RMSE':>14}{'R2':>10}")
for name, m in results.items():
    print(f"{name:<22}{m['MAE']:>14,.2f}{m['MSE']:>18,.2f}{m['RMSE']:>14,.2f}{m['R2']:>10.4f}")


Best model: random_forest_tuned (RMSE=49,531.05, R2=0.8128)

Model                            MAE               MSE          RMSE        R2
linear_regression          49,645.49  4,778,547,424.03     69,127.04    0.6353
random_forest              31,889.92  2,461,153,552.23     49,610.01    0.8122
random_forest_tuned        32,968.48  2,453,324,526.26     49,531.05    0.8128


## Step 10: Predict on new houses (replacing the Flask API)

The original project served this model behind a Flask `/predict` endpoint. In a notebook,
running a live server doesn't make sense — instead we wrap the exact same feature
engineering + `model.predict(...)` logic that the API used into a plain Python function,
`predict_price`, and call it directly. This reproduces the identical prediction logic
without leaving any server or open port running.

We demonstrate it on the same two sample houses the original `scripts/predict.py` used.


In [11]:
def predict_price(model, **fields):
    '''Predict median house value for a single house given its raw fields.

    This mirrors exactly what the Flask API's /predict endpoint did internally:
    build a one-row DataFrame, engineer the same three ratio features, and call
    model.predict(...) on the fitted pipeline.
    '''
    row = pd.DataFrame([fields])
    row["rooms_per_household"] = row["total_rooms"] / row["households"]
    row["bedrooms_per_room"] = row["total_bedrooms"] / row["total_rooms"]
    row["population_per_household"] = row["population"] / row["households"]
    return float(model.predict(row)[0])


sample_houses = [
    {
        "longitude": -122.25, "latitude": 37.85, "housing_median_age": 30.0,
        "total_rooms": 2500.0, "total_bedrooms": 450.0, "population": 900.0,
        "households": 420.0, "median_income": 5.2, "ocean_proximity": "NEAR BAY",
    },
    {
        "longitude": -119.5, "latitude": 36.6, "housing_median_age": 15.0,
        "total_rooms": 1800.0, "total_bedrooms": 350.0, "population": 800.0,
        "households": 300.0, "median_income": 2.8, "ocean_proximity": "INLAND",
    },
]

for i, house in enumerate(sample_houses, start=1):
    price = predict_price(best_model, **house)
    print(f"House {i}: predicted median house value = ${price:,.2f}")


House 1: predicted median house value = $339,816.94
House 2: predicted median house value = $89,763.94


## Step 11: The shape of a live API request/response

To connect this back to the "deploy behind an API" learning objective, here's what a
request/response would look like if this model were served over HTTP — built and
evaluated in plain Python, with no server actually running.


In [12]:
request_body = sample_houses[0]
predicted_value = predict_price(best_model, **request_body)

response_body = {
    "model": best_name,
    "predicted_median_house_value": round(predicted_value, 2),
}

import json
print("Example request body (what a client would POST as JSON):")
print(json.dumps(request_body, indent=2))
print("\nExample response body (what the API would return):")
print(json.dumps(response_body, indent=2))


Example request body (what a client would POST as JSON):
{
  "longitude": -122.25,
  "latitude": 37.85,
  "housing_median_age": 30.0,
  "total_rooms": 2500.0,
  "total_bedrooms": 450.0,
  "population": 900.0,
  "households": 420.0,
  "median_income": 5.2,
  "ocean_proximity": "NEAR BAY"
}

Example response body (what the API would return):
{
  "model": "random_forest_tuned",
  "predicted_median_house_value": 339816.94
}


## Summary

- Cleaned real messy data (207 missing `total_bedrooms` values, handled via median
  imputation — never dropped).
- Engineered three ratio features that materially help both models.
- Compared a simple Linear Regression baseline against a Random Forest, and tuned the
  Random Forest with `RandomizedSearchCV`.
- Evaluated honestly on a held-out test set with MAE / MSE / RMSE / R².
- Reproduced the trained model's predictions for new houses without needing a running
  Flask server — a plain Python function does the same job for a notebook context.

### Extension ideas
- Try `XGBRegressor` and compare against the Random Forest.
- Add polynomial features for `median_income`.
- Plot predicted vs. actual values to visualize error patterns geographically.
